# 03 - Ingeniería de Datos con Pandas


En esta etapa se realiza la carga, limpieza y transformación completa 
del dataset de TechVentas S.A.S. usando Pandas. Se detectan y tratan 
los errores intencionales del dataset como valores nulos, productos en 
blanco y cantidades negativas, dejando los datos limpios y listos para 
responder preguntas de negocio.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

## 3.1 Carga del Dataset


In [2]:
# Cargar el CSV especificando encoding y parseando fechas
tabla1 = pd.read_csv('../data/ventas_techventas.csv', 
                 encoding='latin-1', 
                 parse_dates=['fecha'])

# Corregir nombre con encoding corrupto
tabla1['vendedor'] = tabla1['vendedor'].str.replace('Ana Lï¿½ï¿', 'Ana López', regex=False)

# Explorar el dataset
print(tabla1.info())
print("─" * 50)
print(tabla1.head())

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     str           
 1   fecha            49 non-null     datetime64[us]
 2   cliente_id       49 non-null     str           
 3   cliente_nombre   49 non-null     str           
 4   region           49 non-null     str           
 5   producto         48 non-null     str           
 6   categoria        49 non-null     str           
 7   cantidad         49 non-null     float64       
 8   precio_unitario  49 non-null     float64       
 9   descuento        49 non-null     float64       
 10  vendedor         49 non-null     str           
dtypes: datetime64[us](1), float64(3), str(7)
memory usage: 5.3 KB
None
──────────────────────────────────────────────────
                                            order_id      fecha cliente_id  \
0                 

### Observaciones del dataset

- El dataset tiene 60 registros y 11 columnas
- La mayoría de columnas tienen 49 valores no nulos, lo que significa 
  que hay 11 registros con valores nulos
- La columna producto tiene 48 valores no nulos, es decir 12 nulos
- La fila 2 tiene datos mezclados en la columna order_id, lo que 
  indica un error de formato en el CSV original
- Las columnas cantidad y precio_unitario son float64 cuando 
  deberían ser int y float respectivamente

## 3.2 Limpieza de Datos

In [3]:
# Contar nulos antes de limpiar
print("Nulos antes de limpiar:")
print(tabla1.isna().sum())
print(f"Total registros antes: {len(tabla1)}")

Nulos antes de limpiar:
order_id            0
fecha              11
cliente_id         11
cliente_nombre     11
region             11
producto           12
categoria          11
cantidad           11
precio_unitario    11
descuento          11
vendedor           11
dtype: int64
Total registros antes: 60


In [4]:
# Eliminar filas donde producto sea nulo o esté vacío
tabla1 = tabla1[tabla1['producto'].notna() & (tabla1['producto'] != '')]

# Eliminar filas con cantidades negativas o nulas
tabla1 = tabla1[tabla1['cantidad'] > 0]

# Contar nulos después de limpiar
print("Nulos después de limpiar:")
print(tabla1.isna().sum())
print(f"Total registros después: {len(tabla1)}")
print(f"Registros eliminados: {60 - len(tabla1)}")

Nulos después de limpiar:
order_id           0
fecha              0
cliente_id         0
cliente_nombre     0
region             0
producto           0
categoria          0
cantidad           0
precio_unitario    0
descuento          0
vendedor           0
dtype: int64
Total registros después: 47
Registros eliminados: 13


### Observaciones de la limpieza

- Se eliminaron 13 registros en total
- Se descartaron filas con producto nulo o vacío
- Se descartaron filas con cantidad nula o negativa
- El dataset limpio tiene 47 registros listos para el análisis
- Estrategia elegida: eliminación en vez de imputación, porque los 
  registros con nulos tenían múltiples columnas afectadas simultáneamente, 
  lo que hace inviable imputar valores confiables

## 3.3 Feature Engineering
### Creación de nuevas columnas

In [5]:
# Crear columna ingreso_bruto
tabla1['ingreso_bruto'] = tabla1['cantidad'] * tabla1['precio_unitario']

# Crear columna ingreso_neto
tabla1['ingreso_neto'] = tabla1['ingreso_bruto'] * (1 - tabla1['descuento'])

# Crear columna mes
tabla1['mes'] = tabla1['fecha'].dt.to_period('M')

# Verificar las nuevas columnas
print(tabla1[['cantidad', 'precio_unitario', 'descuento', 'ingreso_bruto', 'ingreso_neto', 'mes']].head())

   cantidad  precio_unitario  descuento  ingreso_bruto  ingreso_neto      mes
0       2.0           1200.0       0.10         2400.0        2160.0  2024-01
1      15.0             25.0       0.00          375.0         375.0  2024-01
3      10.0             85.0       0.10          850.0         765.0  2024-01
4       3.0           1200.0       0.15         3600.0        3060.0  2024-01
5      20.0             95.0       0.00         1900.0        1900.0  2024-01


### Observaciones

- ingreso_bruto es el ingreso antes de aplicar descuentos
- ingreso_neto es el ingreso real después de descuentos
- mes extrae el período mensual de la fecha para poder 
  agrupar por mes en el siguiente paso

## 3.4 Agrupación Mensual y Cumplimiento de Metas

In [6]:
# Agrupar por mes
resumen_mensual = tabla1.groupby('mes').agg(
    total_ingresos=('ingreso_neto', 'sum'),
    total_pedidos=('order_id', 'count')
).reset_index()

print(resumen_mensual)

       mes  total_ingresos  total_pedidos
0  2024-01        11446.00              8
1  2024-02         9725.00              8
2  2024-03        12459.00              9
3  2024-04        14659.00              9
4  2024-05        10924.75              8
5  2024-06         8286.75              5


In [7]:
# Crear tabla de metas mensuales
metas = pd.DataFrame({
    'mes': pd.period_range('2024-01', '2024-06', freq='M'),
    'meta': [30000, 30000, 30000, 30000, 30000, 30000]
})

# Unir resumen mensual con metas usando merge
resumen_completo = resumen_mensual.merge(metas, on='mes')

# Calcular porcentaje de cumplimiento
resumen_completo['pct_cumplimiento'] = round(
    resumen_completo['total_ingresos'] / resumen_completo['meta'] * 100, 2
)

print(resumen_completo)

       mes  total_ingresos  total_pedidos   meta  pct_cumplimiento
0  2024-01        11446.00              8  30000             38.15
1  2024-02         9725.00              8  30000             32.42
2  2024-03        12459.00              9  30000             41.53
3  2024-04        14659.00              9  30000             48.86
4  2024-05        10924.75              8  30000             36.42
5  2024-06         8286.75              5  30000             27.62


### Observaciones

- Ningún mes alcanzó el 50% de cumplimiento de la meta mensual
- Abril fue el mejor mes con 48.86% de cumplimiento
- Junio fue el peor mes con solo 27.62%
- Se observa una tendencia de caída en los últimos dos meses 
  del semestre, lo que podría ser una señal de alerta para el CEO

## 3.5 Exportar a SQLite y Verificar


In [8]:
# Conectar a SQLite
conn = sqlite3.connect('../output/techventas.db')

# Convertir columna mes a texto para que SQLite la acepte
tabla1['mes'] = tabla1['mes'].astype(str)

# Exportar el dataframe limpio a SQLite
tabla1.to_sql('ventas', conn, if_exists='replace', index=False)

# Verificar leyendo de vuelta
verificacion = pd.read_sql('SELECT COUNT(*) AS total_registros FROM ventas', conn)
print(verificacion)

   total_registros
0               47


### Observaciones

- Se exportaron 47 registros limpios a la base de datos SQLite
- La verificación con read_sql confirma que los datos 
  se guardaron correctamente
- La columna mes se convirtió a texto para compatibilidad con SQLite

In [9]:
# Cerrar conexión
conn.close()
print("Conexión cerrada")

Conexión cerrada


## Conclusiones Finales - Reporte para el CEO

### TechVentas S.A.S - Análisis Primer Semestre 2024

**1. Ingresos mensuales y cumplimiento de metas**
Ningún mes alcanzó el 50% de cumplimiento de la meta mensual de $30,000.
Abril fue el mejor mes con $14,659 y 48.86% de cumplimiento, mientras que 
Junio fue el peor con $8,286 y solo 27.62%. Se observa una tendencia 
preocupante de caída en los últimos meses del semestre.

**2. Rendimiento por vendedor**
Carlos Ruiz lideró el semestre con $21,355 seguido de Ana López con $20,810.
Maria Torres tuvo el menor desempeño con $11,860. Ningún vendedor superó 
el 50% de su meta semestral, lo que sugiere revisar si las metas son 
realistas o si el equipo necesita refuerzo comercial.

**3. Cliente más valioso**
TechCorp SA (C001) fue el cliente que generó el mayor ingreso total 
del semestre con $19,025. Es un cliente estratégico que merece 
atención prioritaria.

**4. Categorías del catálogo**
Almacenamiento fue la categoría con mayor volumen de unidades vendidas (260),
mientras que Electrónica tiene el precio promedio más alto ($1,139), 
lo que la convierte en la categoría premium del catálogo.

**5. Calidad de los datos**
Se detectaron y eliminaron 13 registros con errores en el dataset original,
incluyendo valores nulos, productos en blanco y cantidades negativas. 
El dataset limpio quedó con 47 registros confiables para el análisis.